In [1]:
import re
import pandas as pd
import os
import scholarly
import time
from utils.utils import ScholarlyPublication
from scholarly import scholarly, ProxyGenerator


In [2]:
pg = ProxyGenerator()
# pg.ScraperAPI('f859b36e57b1a9e2403fc0a263f6c5d6')
pg.FreeProxies()
scholarly.use_proxy(pg)

In [3]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]


In [ ]:
def compile_query(keywords_inclusion, keywords_exclusion, journals_to_include):
    query = "("
    for idx, keyword in enumerate(keywords_inclusion):
        if idx == 0:
            query += f"{keyword} "
        else:
            query += f"OR {keyword} "
    query += ") AND ("
    
    for idx, journal in enumerate(journals_to_include):
        if idx == 0:
            query += f" source:{journal} "
        else:
            query += f"OR source:{journal} "
    query += ") "
    for idx, keyword in enumerate(keywords_exclusion):
        query += f"-{keyword} "

    return query
# query = compile_query(keywords_inclusion, keywords_exclusion, journals_to_include)

In [ ]:
def match_journal(article_title, article_journal):
    for journal in journals_to_include:
        if re.search(journal, article_journal, re.IGNORECASE):
            return True
    return False

def match_inclusion_criteria(article_title, article_abstract):
    for keyword in keywords_inclusion:
        if re.search(keyword, article_title, re.IGNORECASE) or re.search(keyword, article_abstract, re.IGNORECASE):
            return True
    return False

def match_exclusion_criteria(article_title, article_abstract):
    for keyword in keywords_exclusion:
        if re.search(keyword, article_title, re.IGNORECASE) or re.search(keyword, article_abstract, re.IGNORECASE):
            return True
    return False

def search_articles(query):
    search_query = scholarly.search_pubs(query)
    articles = []

    print("Searching for articles...")
    count = 0
    
    for article in search_query:
        print(f"{count+1} articles")
        count += 1
        # if (match_journal(article['bib']['title'], article['bib']['venue']) and
        #         match_inclusion_criteria(article['bib']['title'], article['bib'].get('abstract', 'N/A')) and
        #         not match_exclusion_criteria(article['bib']['title'], article['bib'].get('abstract', 'N/A'))):
        articles.append(article)
        time.sleep(2)
            

    return articles



In [4]:
# articles = search_articles(query)
query = ' OR '.join(keywords_inclusion)

In [6]:
search_query = scholarly.search_pubs(query)
articles = []


In [7]:

print("Searching for articles...")
count = 0

for article in search_query:
    print(f"{count+1} articles")
    count += 1
    # sometimes this times out, so we need to catch the exception
    
    paper = article.fill()
    articles.append(paper)
    time.sleep(1)

Searching for articles...
1 articles


AttributeError: 'dict' object has no attribute 'fill'

In [ ]:
len(articles)

In [ ]:
articles[0]

In [ ]:
# create a ScholarlyPublication() from each aritcle and store to json and json file

output_path = r"C:\Users\20195435\OneDrive - TU Eindhoven\TUe\Playground\Nanotechnology\scholarly_articles"
temp_pub = ScholarlyPublication()
publications = []
for article in articles:
    print(article['bib'])

    # temp_pub = ScholarlyPublication()
    temp_article = temp_pub.load_from_json(article)

    print(temp_article)
    
    temp_name = sanitize_filename(temp_article.bib['title'])
    article_filename = temp_name +".json"
    article_filepath = os.path.join(output_path, article_filename)
    temp_article.save_to_json(article_filepath)

    publications.append(temp_article)
